# FailureSensorIQ GEPA Optimization

- In this experiment we utilize open source models like `gpt-oss-120b` to optimize Chain of Thought (CoT) `dspy.ChainOfThought` for solving the `FailureSensorIQ` Datasets MCQA results.   

In [1]:
# Library Imports
import os
from datasets import load_dataset
import dspy
from dspy import GEPA

import json
import random
import re
import textwrap
from typing import Optional

In [2]:
# Global Variables
seed = 42
n_train_assets = 2


In [3]:
# Watsonx AI Credentials
from ibm_watsonx_ai import APIClient,Credentials
api_key = os.getenv("WATSONX_API_KEY")
url = "https://us-south.ml.cloud.ibm.com"
project_id = os.getenv("WATSONX_PROJECT_ID")

#Testing Credentials 
credentials = Credentials(api_key=api_key, url=url)

client = APIClient(credentials)

client.set.default_project(project_id)

'SUCCESS'

In [4]:
# Setup DSPy

import dspy
model_name = "watsonx/openai/gpt-oss-120b"
max_tokens = 32000
temperature = 1

lm = dspy.LM(model=model_name, max_tokens=max_tokens, api_key=api_key,temperature = temperature
             ,project_id = project_id
             ,api_base=url
             )

dspy.configure(lm=lm, trace=[], experimental=True)

In [5]:
# Helper Functions

def _parse_answer(option_ids: list[str], correct: list[bool]) -> str:
    """
    Convert the dataset's boolean correct list into a letter string.

    Example:
        option_ids = ["A", "B", "C", "D", "E"]
        correct    = [False, False, False, True, False]
        → "D"

        option_ids = ["A", "B", "C"]
        correct    = [True, False, True]
        → "A,C"
    """
    letters = [lid.upper() for lid, flag in zip(option_ids, correct) if flag]
    return ",".join(sorted(letters))


def _build_options_str(options: list[str], option_ids: list[str]) -> str:
    """
    Render options as a lettered multi-line string.

    Example:
        options    = ["partial discharge", "resistance", "current"]
        option_ids = ["A", "B", "C"]
        →  "  A) partial discharge\n  B) resistance\n  C) current"
    """
    return "\n".join(
        f"  {lid}) {text}"
        for lid, text in zip(option_ids, options)
    )


def _collect_and_shuffle(assets: list[str], data_dict: dict[str, list[dspy.Example]]) -> list[dspy.Example]:

    rng = random.Random(seed)
    combined = [ex for a in assets for ex in data_dict.get(a, [])]
    rng.shuffle(combined)
    return combined

## Load FailureSensorIQ Dataset

In [6]:
 # Load FailureSensorIQ Dataset

ds = load_dataset("ibm-research/FailureSensorIQ", "single_true_multi_choice_qa")


In [7]:

def init_dataset_splits(ds):

    fsiq_dict: dict[str, list[dspy.Example]] = {}

    # Assets used for Training
    n_train_assets = 2

    # Parsing Rows into dspy.Examples
    for row in ds["train"]:

        q_type = row.get("question_type", "")

        options_str = _build_options_str(row["options"], row["option_ids"])
        answer      = _parse_answer(row["option_ids"], row["correct"])
        asset       = row.get("asset_name", "unknown")

        ex = dspy.Example(
            question      = row["question"],
            options       = options_str,
            answer        = answer,
            asset         = asset,
            query_type    = row.get("relevancy", "unknown"),
            question_type = q_type,
        ).with_inputs("question", "options")
    
        fsiq_dict.setdefault(asset, []).append(ex)

    train_assets     = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i < n_train_assets]
    remaining_assets = [asset for i, (asset, examples) in enumerate(fsiq_dict.items()) if i >= n_train_assets]


    val_assets  = [a for i, a in enumerate(remaining_assets) if i % 2 == 0]
    test_assets = [a for i, a in enumerate(remaining_assets) if i % 2 == 1]

    print("train_assets :" , train_assets)
    print("val_assets   :" , val_assets)
    print("test_assets  :" , test_assets)

    train_set = _collect_and_shuffle(train_assets, fsiq_dict)
    val_set   = _collect_and_shuffle(val_assets, fsiq_dict)
    test_set  = _collect_and_shuffle(test_assets, fsiq_dict)*3 #*5  

    print(f"train_set: {len(train_set)} examples")
    print(f"val_set: {len(val_set)} examples")
    print(f"test_set: {len(test_set)} examples")

    return train_set, val_set, test_set

In [8]:
train_set, val_set, test_set = init_dataset_splits(ds)

train_assets : ['electric motor', 'steam turbine']
val_assets   : ['aero gas turbine', 'pump', 'reciprocating internal combustion engine', 'fan']
test_assets  : ['industrial gas turbine', 'compressor', 'electric generator', 'power transformer']
train_set: 405 examples
val_set: 1024 examples
test_set: 3714 examples


## `dspy.chainofthought` Setup

In [9]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    question  : str = dspy.InputField(desc="The MCQA question about failure modes or sensors.")
    options   : str = dspy.InputField(desc="Lettered answer choices, one per line.")
    reasoning : str = dspy.OutputField(desc="Step-by-step reasoning referencing FMEA knowledge.")
    answer    : str = dspy.OutputField(desc="Correct letter(s), e.g. 'D' or 'A,C'.")

program = dspy.ChainOfThought(GenerateResponse)



### Defining The Evaluation Metric

In [10]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        return 0
    
    return int(correct_answer_normalized == llm_answer_normalized)

### Evaluating the Unoptimized Baseline Chain of Thought (CoT)

In [11]:
evaluate = dspy.Evaluate(
    devset=test_set, #change to test_set for final evaluation
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

result_base = evaluate(program)

Average Metric: 658.00 / 991 (66.4%):  27%|██▋       | 990/3714 [02:05<05:12,  8.71it/s]

2026/03/06 22:31:00 ERROR dspy.utils.parallelizer: Error for Example({'question': 'If the sensor partial discharge in power transformer shows an abnormal reading, which failure event is insignificant?', 'options': '  A) connection/ bushing faults\n  B) insulation deterioration\n  C) winding distortion\n  D) on-load tap-changer condition/ fault\n  E) de-energized tap-changer condition/ fault', 'answer': 'C', 'asset': 'power transformer', 'query_type': 'irrelevant_failure_modes_for_sensor', 'question_type': 'mcp1_negative'}) (input_keys={'question', 'options'}): litellm.RateLimitError: WatsonxException - {"errors":[{"code":"rate_limit_reached_requests","message":"Rate limit of 8 requests per 1s was reached for instance id 2893a0d1-ecf8-4e12-82c1-90b6c294a4ed (user IBMid-2700001QEW, plan a3d2f92f-06f9-48d0-b2e6-a7ba2b4e0577)","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-chat"}],"trace":"3c3273624ff74ed8ebd44d0102e76353","status_code":429}. Set `provide_traceback=True` for tr

Average Metric: 2473.00 / 3713 (66.6%): 100%|██████████| 3714/3714 [02:48<00:00, 22.00it/s] 


2026/03/06 22:31:43 INFO dspy.evaluate.evaluate: Average Metric: 2473.0 / 3714 (66.6%)


,question,options,example_answer,asset,query_type,question_type,reasoning,pred_answer,metric,answer
0,"For power transformer, if a failure event core looseness occurs, w...",A) ultrasound\n B) noise\n C) resistance\n D) vibration,C,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,In a Failure Mode and Effects Analysis (FMEA) for power transforme...,C,✔️ [1.000],NaN
1,What is the irrelevant failure event for electric generator if the...,A) bearing damage\n B) unbalance,B,electric generator,irrelevant_failure_modes_for_sensor,mcp1_negative,"In an FMEA for an electric generator, the “coast‑down” sensor is u...",B,✔️ [1.000],NaN
2,"When considering connection/ bushing faults in power transformer, ...",A) power factor/tanδ\n B) noise\n C) temperature\n D) resista...,E,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,Connection or bushing faults in a power transformer are primarily ...,B,✔️ [0.000],NaN
3,Which sensor out of the choices is not effective in indicating the...,A) temperature\n B) noise\n C) oil condition\n D) bushing cap...,D,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,To detect a low oil level in a power transformer we need a paramet...,C,✔️ [0.000],NaN
4,"In industrial gas turbine, which failure event is unimportant if t...",A) compressor damaged\n B) compressor fouled\n C) power turbin...,C,industrial gas turbine,irrelevant_failure_modes_for_sensor,mcp1_negative,The air‑flow sensor is located at the inlet of the gas‑turbine and...,C,✔️ [1.000],NaN
...,...,...,...,...,...,...,...,...,...,...
3709,When the sensor oil leakage in compressor displays an abnormal rea...,A) bearing damage\n B) compressor stall,B,compressor,irrelevant_failure_modes_for_sensor,mcp1_negative,An oil‑leakage sensor in a compressor monitors the presence of oil...,B,✔️ [1.000],NaN
3710,Which failure mode is most relevant for electric generator if ther...,A) insulation deterioration\n B) stator windings fault\n C) mi...,A,electric generator,relevant_failure_modes_for_sensor,mcp1_positive,Abnormal voltage readings in a generator are directly related to t...,B,✔️ [0.000],NaN
3711,Which sensor out of the choices does not indicate the presence of ...,A) torque\n B) temparature\n C) vibration\n D) oil debris\n ...,E,electric generator,irrelevant_sensors_for_failure_mode,mcp1_negative,Bearing damage in rotating equipment such as an electric generator...,E,✔️ [1.000],NaN
3712,"In industrial gas turbine, which failure mode is most important if...",A) air inlet blockage\n B) power turbine damaged\n C) compress...,C,industrial gas turbine,relevant_failure_modes_for_sensor,mcp1_positive,Abnormal readings from the fuel‑pressure / fuel‑flow sensors indic...,C,✔️ [1.000],NaN


## GEPA Optimization

In [12]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Metric that checks if prediction is one of the valid options (A-E or combinations like A,C)
    and matches the correct answer.
    """
    correct_answer = example['answer']
    
    try:
        llm_answer = str(prediction.answer).strip().upper()
        
        # Validate that answer is one of the valid options (single letter or comma-separated letters A-E)
        # Valid formats: "A", "B", "A,C", "B,D", etc.
        answer_pattern = r'^[A-E](,[A-E])*$'
        
        if not re.match(answer_pattern, llm_answer):
            return 0
        
        # Ensure answers are sorted (e.g., "A,C" not "C,A") to match expected format
        parts = llm_answer.split(',')
        llm_answer_normalized = ','.join(sorted(set(parts)))
        correct_answer_normalized = ','.join(sorted(set(correct_answer.split(','))))
        
    except (ValueError, AttributeError, TypeError):
        feedback_text = f"The final answer must be one of the options (A-E or combinations like A,C). You responded with '{prediction.answer}', which couldn't be parsed as a valid option. Please ensure your answer is a valid option without any additional text or formatting."
        feedback_text += f" The correct answer is '{correct_answer}'."      

        #print(feedback_text)

        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)

    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    return int(correct_answer_normalized == llm_answer_normalized)

In [13]:
optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=3,
    reflection_lm=dspy.LM(model=model_name, max_tokens=max_tokens, api_key=api_key,tenperature = temperature
             ,project_id = project_id   
             ,api_base=url)
)

optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set
)

2026/03/06 22:31:43 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 4476 metric calls of the program. This amounts to 3.13 full evals on the train+val set.
2026/03/06 22:31:43 INFO dspy.teleprompt.gepa.gepa: Using 1024 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/4476 [00:00<?, ?rollouts/s]2026/03/06 22:33:52 INFO dspy.evaluate.evaluate: Average Metric: 634 / 1024 (61.9%)
2026/03/06 22:33:52 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.619140625 over 1024 / 1024 examples
GEPA Optimization:  23%|██▎       | 1024/4476 [02:09<07:16,  7.90rollouts/s]2026/03/06 22:33:52 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 

Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:02<00:00,  1.12it/s]

2026/03/06 22:33:55 INFO dspy.evaluate.evaluate: Average Metric: 3 / 3 (100.0%)
2026/03/06 22:33:55 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/03/06 22:33:55 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:  23%|██▎       | 1027/4476 [02:12<07:27,  7.71rollouts/s]2026/03/06 22:33:55 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.619140625



Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:02<00:00,  1.08it/s]

2026/03/06 22:33:58 INFO dspy.evaluate.evaluate: Average Metric: 0 / 3 (0.0%)


...

2026/03/06 22:47:31 INFO dspy.evaluate.evaluate: Average Metric: 1 / 3 (33.3%)


2026/03/06 22:47:40 INFO dspy.teleprompt.gepa.gepa: Iteration 37: Proposed new text for predict: **Task Overview**

You will be given a prompt that contains two parts:

1. **question** – a single‑sentence or short‑paragraph query about a piece of engineering equipment (e.g., electric motors, steam turbines, etc.).  
2. **options** – a list of possible answer choices, each prefixed with a capital letter (A, B, C, …).

Your job is to determine which option best satisfies the request in the question and to present both a concise justification and the final choice.

**Output Format**

Your response must contain **exactly two sections**, in this order:

```
### reasoning
<brief, fact‑based explanation of why each option is evaluated and why the selected one is the correct answer>

### answer
<single capital letter of the chosen option>
```

Do **not** include any additional headings, bullet points, or extraneous text outside the two sections above.

**General Reasoning Procedure**

1. **Par

Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:03<00:00,  1.31s/it] 

2026/03/06 22:47:45 INFO dspy.evaluate.evaluate: Average Metric: 1 / 3 (33.3%)


2026/03/06 22:48:04 INFO dspy.teleprompt.gepa.gepa: Iteration 38: Proposed new text for predict: **Task Overview**  
You will receive:

1. **A single‑sentence question** about either  
   *Steam‑turbine (mechanical) failures* **or** *Electric‑motor diagnostics*.

2. **A list of answer options** labelled **A, B, C, …** (one per line).

Your job is to **select the single option that correctly answers the question** and **output only the letter** (e.g., `C`).  
Do **not** add any extra characters, spaces, line‑breaks, punctuation, or markdown.

---

## 1. How to recognise the request type  

Examine the wording of the question and classify it into one of the four request types below.  
The classification tells you exactly what you must choose.

| Keywords / phrasing in the question                               | What you must select                                                     |
|-------------------------------------------------------------------|----------------------------------

# Checking the generated prompt

In [14]:
print(optimized_program.predict.signature.instructions)

**Task Overview**

You will be given a short question about turbomachinery (steam turbines) or electric motors, followed by a list of multiple‑choice options (labeled A, B, C, …).  
Your job is to pick the *single* option that correctly answers the question and output **only the letter** of that option (e.g., `A`). No extra text, no explanations, and no markdown should be included in the final response.

**How to Solve the Problems**

1. **Read the question carefully** to understand which of the following is being asked:
   - *“most relevant sensor regarding the occurrence of the failure event”* → choose the sensor that best detects **that** failure.
   - *“most relevant failure mode … when resistance detects abnormal readings”* → choose the failure mode that is most directly indicated by an **abnormal winding resistance** measurement.
   - *“which sensor … is not effective in indicating the presence of …”* → choose the sensor that **does not** give a reliable indication of the listed 

Evaluating the Optimized Program 

In [15]:
result_optimized = evaluate(optimized_program)

Average Metric: 408.00 / 570 (71.6%):  15%|█▌        | 570/3714 [01:11<06:20,  8.27it/s]

2026/03/06 22:52:13 ERROR dspy.utils.parallelizer: Error for Example({'question': 'What is the irrelevant failure event for electric generator if the sensor power exhibits an abnormal reading?', 'options': '  A) brush(es) fault\n  B) insulation deterioration', 'answer': 'B', 'asset': 'electric generator', 'query_type': 'irrelevant_failure_modes_for_sensor', 'question_type': 'mcp1_negative'}) (input_keys={'question', 'options'}): litellm.RateLimitError: WatsonxException - {"errors":[{"code":"rate_limit_reached_requests","message":"Rate limit of 8 requests per 1s was reached for instance id 2893a0d1-ecf8-4e12-82c1-90b6c294a4ed (user IBMid-2700001QEW, plan a3d2f92f-06f9-48d0-b2e6-a7ba2b4e0577)","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-chat"}],"trace":"066d2e743db77071c42ac0c64dd98369","status_code":429}. Set `provide_traceback=True` for traceback.


Average Metric: 2574.00 / 3713 (69.3%): 100%|██████████| 3714/3714 [02:44<00:00, 22.52it/s] 


2026/03/06 22:53:46 INFO dspy.evaluate.evaluate: Average Metric: 2574.0 / 3714 (69.3%)


,question,options,example_answer,asset,query_type,question_type,reasoning,pred_answer,metric,answer
0,"For power transformer, if a failure event core looseness occurs, w...",A) ultrasound\n B) noise\n C) resistance\n D) vibration,C,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,Core looseness in a power transformer is a mechanical fault that t...,C,✔️ [1.000],NaN
1,What is the irrelevant failure event for electric generator if the...,A) bearing damage\n B) unbalance,B,electric generator,irrelevant_failure_modes_for_sensor,mcp1_negative,The coast‑down sensor records the rate at which the generator’s ro...,B,✔️ [1.000],NaN
2,"When considering connection/ bushing faults in power transformer, ...",A) power factor/tanδ\n B) noise\n C) temperature\n D) resista...,E,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,Connection or bushing faults in a power transformer affect the ins...,E,✔️ [1.000],NaN
3,Which sensor out of the choices is not effective in indicating the...,A) temperature\n B) noise\n C) oil condition\n D) bushing cap...,D,power transformer,irrelevant_sensors_for_failure_mode,mcp1_negative,Low oil level in a power transformer primarily manifests through t...,B,✔️ [0.000],NaN
4,"In industrial gas turbine, which failure event is unimportant if t...",A) compressor damaged\n B) compressor fouled\n C) power turbin...,C,industrial gas turbine,irrelevant_failure_modes_for_sensor,mcp1_negative,An abnormal reading from the air‑flow sensor indicates that the am...,C,✔️ [1.000],NaN
...,...,...,...,...,...,...,...,...,...,...
3709,When the sensor oil leakage in compressor displays an abnormal rea...,A) bearing damage\n B) compressor stall,B,compressor,irrelevant_failure_modes_for_sensor,mcp1_negative,An oil‑leakage sensor signals that oil is escaping from the compre...,B,✔️ [1.000],NaN
3710,Which failure mode is most relevant for electric generator if ther...,A) insulation deterioration\n B) stator windings fault\n C) mi...,A,electric generator,relevant_failure_modes_for_sensor,mcp1_positive,Abnormal voltage readings in an electric generator indicate a prob...,A,✔️ [1.000],NaN
3711,Which sensor out of the choices does not indicate the presence of ...,A) torque\n B) temparature\n C) vibration\n D) oil debris\n ...,E,electric generator,irrelevant_sensors_for_failure_mode,mcp1_negative,Bearing damage in an electric generator is typically monitored by ...,A,✔️ [0.000],NaN
3712,"In industrial gas turbine, which failure mode is most important if...",A) air inlet blockage\n B) power turbine damaged\n C) compress...,C,industrial gas turbine,relevant_failure_modes_for_sensor,mcp1_positive,Abnormal readings of fuel pressure or fuel flow indicate that the ...,C,✔️ [1.000],NaN


We see an improvement from the unoptimized score of 
- **66.6%%** to **69.3%** (Temp = 1)